# WS4 — GRPO Training (Path A: Unsloth + TRL)

FarmSimulation hackathon, Person B. Trains `Qwen2.5-0.5B-Instruct` on Task 1 via TRL `GRPOTrainer` with Unsloth's vLLM-backed fast inference and 4-bit + LoRA.

**Status:** scaffold — cells 1–4 only. Cells 5–13 (FarmEnvClient, parse_action, 5 reward functions, GRPOConfig, trainer.train, plots, Hub upload) are pending HANDOFF #1 (A's WS1 merge — adds `narrative_text` to `FarmObservation`). Reference: `IMPLEMENTATION_PLAN.md` §20.3 / §20.7.

**Runtime:** Colab T4 (free tier OK for cells 1–4 smoke test). Final 50-iter GRPO training will run on the dedicated L4 Space at H+18-19.

**Critical flags (do not change without re-reading §20.3):**
- `fast_inference=True` requires `load_in_4bit=True` (vLLM backend, ~10× generation speedup).
- `use_gradient_checkpointing="unsloth"` — **string**, not `True`. Unsloth's custom impl saves an extra ~30% VRAM.
- `gpu_memory_utilization=0.7` — drop to `0.5` only if you OOM during rollout.

**Pin deviation from §20.7:** the plan's `trl==0.11.4` pin predates DAPO loss (TRL ≥0.13) and two-sided clipping (TRL ≥0.15) which §20.5 requires, and triggers a `UnslothGKDTrainer` SyntaxError under current `unsloth_zoo`. Cell 1 below pins `trl==0.22.2` + `transformers==4.56.2` to match Unsloth's official `Qwen2.5_(3B)-GRPO.ipynb` (April 2026, github.com/unslothai/notebooks).

In [1]:
!pip install -q unsloth vllm \
trl==0.22.2 \
transformers==4.56.2 \
huggingface_hub==0.34.0 \
openenv-core wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.7/558.7 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import importlib.metadata as _md

for _pkg in ["unsloth", "unsloth_zoo", "trl", "transformers", "torch", "vllm", "huggingface_hub"]:
    try:
        print(f"{_pkg:20} = {_md.version(_pkg)}")
    except:
        print(f"{_pkg:20} = NOT INSTALLED")

unsloth              = 2026.4.8
unsloth_zoo          = 2026.4.9
trl                  = 0.22.2
transformers         = 4.56.2
torch                = 2.10.0+cu128
vllm                 = 0.19.1
huggingface_hub      = 0.34.0


In [3]:
from huggingface_hub import login as hf_login
hf_login()

import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: senv7587 (senv7587-scaler) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
    max_seq_length=1104,                # = max_prompt_length (1024) + max_completion_length (80)
    load_in_4bit=True,                  # 4-bit base, LoRA stays bf16
    fast_inference=True,                # vLLM backend → ~10× faster generation
    max_lora_rank=16,
    gpu_memory_utilization=0.7,         # leave 30% for activations + KV cache
    enforce_eager=True
)

INFO 04-25 15:30:24 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.19.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit with actual GPU utilization = 67.06%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 1104. Num Sequences = 48.
Unsloth: vLLM's KV Cache can use up to 9.14 GB. Also swap space = 0 GB.
Unsloth: Not an error, but `level` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but 

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 04-25 15:30:39 [gpu_model_runner.py:4820] Model loading took 0.44 GiB memory and 1.830370 seconds
INFO 04-25 15:33:15 [gpu_worker.py:436] Available KV cache memory: 9.15 GiB
INFO 04-25 15:33:15 [kv_cache_utils.py:1319] GPU KV cache size: 799,504 tokens
INFO 04-25 15:33:15 [kv_cache_utils.py:1324] Maximum concurrency for 1,104 tokens per request: 724.19x
INFO 04-25 15:33:15 [kernel_warmup.py:69] Warming up FlashInfer attention.
INFO 04-25 15:35:27 [core.py:283] init engine (profile, create kv cache, warmup model) took 287.46 seconds
Unsloth: Just some info: will skip parsing ['post_layernorm', 'layer_norm1', 'post_feedforward_layernorm', 'norm', 'input_layernorm', 'layer_norm2', 'ffn_norm', 'q_norm', 'attention_norm', 'k_norm', 'norm1', 'norm2', 'pre_feedforward_layernorm', 'post_attention_layernorm']


Some weights of Qwen2ForCausalLM were not initialized from the model checkpoint at unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,                      # alpha = 2 × r is Unsloth's default rule
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",   # NOT True/False — the string "unsloth" enables their custom impl
    random_state=3407,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.8 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
